# SHARP-LLM · ICMLDE Five-Seed Reproducibility — Kaggle Runner
> Implements `context/journal-paper/icmlde-five-seed-reproducibility-plan.md`  
> Designed for **Kaggle T4 GPU** — fully CLI-driven, no browser needed after first push.

## CLI workflow (run from your local machine)

```bash
# 1. Edit VARIANT in Cell 4, then push:
kaggle kernels push -p colab/

# 2. Poll until complete:
kaggle kernels status msbasanth/sharp-llm-icmlde-runner

# 3. Download outputs:
kaggle kernels output msbasanth/sharp-llm-icmlde-runner -p outputs/icmlde2026/

# 4. Re-upload outputs for next session's resume capability:
kaggle datasets version -p outputs/icmlde2026/ -m "Session N: <variant>"

# 5. Change VARIANT and push again.
```

## Four sessions on Kaggle free tier (30 GPU-hrs/week)

| Session | `VARIANT` | Est. T4 time | Kaggle 9h limit |
|---------|-----------|-------------|------------------|
| 1 | `codet5-small` | ~2 h | ✅ |
| 2 | `codet5-base` | ~7 h | ✅ |
| 3 | `codebert-base` | ~8 h | ⚠ tight |
| 4 | `graphcodebert-base` | ~8 h | ⚠ tight |

**Persistence model:** Kaggle working dir is ephemeral. Download outputs after each session and re-upload as `msbasanth/sharp-llm-icmlde-outputs` (see Cell 13).

In [ ]:
# ── Cell 2 · GPU check ────────────────────────────────────────────────────────
import subprocess, torch

# nvidia-smi is informational only — torch CUDA is the authoritative check
try:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if r.returncode == 0:
        print(r.stdout)
    else:
        print("nvidia-smi returned non-zero (GPU may still be available via torch)")
except (FileNotFoundError, OSError) as e:
    print(f"nvidia-smi not in PATH ({e}) — checking torch CUDA directly ...")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected by torch.\n"
        "On Kaggle: Settings (right panel) -> Accelerator -> GPU T4 x1, then Save & Run All."
    )

p = torch.cuda.get_device_properties(0)
print(f"GPU  : {p.name}")
print(f"VRAM : {p.total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")


In [ ]:
# -- Cell 3 . Paths --
import os
from pathlib import Path

REPO_DIR          = "/kaggle/working/sharp-llm"
DATA_INPUT        = "/kaggle/input/sharp-llm-processed-data"
PREV_OUTPUT_INPUT = ""    # Session 1: empty. Sessions 2+: "/kaggle/input/sharp-llm-icmlde-outputs"
OUTPUT_ROOT       = "/kaggle/working/icmlde2026/juliet118"

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("REPO_DIR          :", REPO_DIR)
print("DATA_INPUT        :", DATA_INPUT)
print("PREV_OUTPUT_INPUT :", PREV_OUTPUT_INPUT or "(empty - session 1)")
print("OUTPUT_ROOT       :", OUTPUT_ROOT)
print("Cell 3 done.")


In [ ]:
# ── Cell 4 · Session config — EDIT THIS CELL EACH PUSH ───────────────────────
# Plan §Variable Under Study: only the training seed changes between runs.

# ▶ Change for each `kaggle kernels push`:
#   codet5-small | codet5-base | codebert-base | graphcodebert-base
VARIANT = "codet5-small"

# All 5 plan seeds — runner skips already-completed ones automatically
SEEDS = "42,43,44,45,46"

# Repo URL — use PAT token for private repos:
#   https://<PAT>@github.com/msbasanth/sharp-llm.git
REPO_URL = "https://github.com/msbasanth/sharp-llm.git"

print(f"VARIANT : {VARIANT}")
print(f"SEEDS   : {SEEDS}")
print(f"REPO    : {REPO_URL}")

In [ ]:
# ── Cell 5 · Clone repo + install dependencies ───────────────────────────────
import os, subprocess, sys

REPO_DIR = "/kaggle/working/sharp-llm"

if os.path.isdir(f"{REPO_DIR}/.git"):
    os.system(f"git -C {REPO_DIR} pull --quiet")
    print("Repo updated.")
else:
    ret = os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError("git clone failed — check REPO_URL in Cell 4.")

os.chdir(REPO_DIR)

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    check=False,
)
if ret.returncode != 0:
    raise RuntimeError("pip install failed — check requirements.txt")

print(f"✓ Repo ready at {REPO_DIR}")
print("✓ Dependencies installed")
# -- P100 compatibility fix: reinstall PyTorch after requirements.txt --
# requirements.txt may install a PyTorch version incompatible with P100 (sm_60).
# This block detects P100 and reinstalls a compatible version.
import subprocess as _sp, sys as _sys
_nv = _sp.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
               capture_output=True, text=True)
if _nv.returncode == 0 and _nv.stdout.strip().startswith("6."):
    print(f"P100 detected (sm_{_nv.stdout.strip()}) -- reinstalling PyTorch 2.0.1+cu118 ...")
    _r = _sp.run([_sys.executable, "-m", "pip", "install",
                  "torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
                  "--index-url", "https://download.pytorch.org/whl/cu118",
                  "--quiet"], capture_output=False)
    if _r.returncode == 0:
        print("PyTorch 2.0.1+cu118 installed -- P100 now supported.")
    else:
        print("WARNING: PyTorch reinstall failed -- training will error on P100.")
else:
    print("GPU compute cap:", _nv.stdout.strip() if _nv.returncode == 0 else "unknown", "-- no reinstall needed.")


### First-time only — create the two Kaggle datasets

Run these commands **once** from your local machine before the first push:

```bash
# ── Dataset 1: processed training data ───────────────────────────────────────
mkdir -p kaggle-data-upload
cp data/processed/train.parquet   kaggle-data-upload/
cp data/processed/test.parquet    kaggle-data-upload/
cp data/processed/label_map.json  kaggle-data-upload/

# Create metadata
echo '{"title":"sharp-llm-processed-data","id":"msbasanth/sharp-llm-processed-data","licenses":[{"name":"other"}]}' \
  > kaggle-data-upload/dataset-metadata.json

kaggle datasets create -p kaggle-data-upload/

# ── Dataset 2: outputs placeholder (empty, updated after each session) ────────
mkdir -p kaggle-outputs-upload
echo '{"title":"sharp-llm-icmlde-outputs","id":"msbasanth/sharp-llm-icmlde-outputs","licenses":[{"name":"other"}]}' \
  > kaggle-outputs-upload/dataset-metadata.json
echo '{}' > kaggle-outputs-upload/placeholder.json

kaggle datasets create -p kaggle-outputs-upload/
```

After session 1 completes (Cell 13 shows download instructions).

In [ ]:
# -- Cell 6 . Verify data and link outputs dir --
# Data comes from the git clone (parquets are now committed to the repo).
import os
from pathlib import Path

REPO_DIR    = "/kaggle/working/sharp-llm"
OUTPUT_ROOT = "/kaggle/working/icmlde2026/juliet118"

os.chdir(REPO_DIR)

# Verify data files from git clone
for name in ["data/processed/train.parquet", "data/processed/test.parquet", "data/processed/label_map.json"]:
    p = Path(name)
    if p.exists():
        print(f"OK  {name}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        raise FileNotFoundError(f"Missing: {name} -- ensure parquets are committed to the repo")

# Link outputs dir -> /kaggle/working (writable across cells)
out_dst = Path(REPO_DIR) / "outputs" / "icmlde2026"
out_src = Path(OUTPUT_ROOT)
out_src.mkdir(parents=True, exist_ok=True)
if not out_dst.exists():
    out_dst.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(out_src), str(out_dst))
    print(f"Linked outputs/icmlde2026 -> {out_src}")
else:
    print("outputs/icmlde2026 already present")
print("Cell 6 done.")


In [ ]:
# -- Cell 7 . Phase 1 -- data preflight + manifest check --
import json, os
from pathlib import Path

REPO_DIR = "/kaggle/working/sharp-llm"
os.chdir(REPO_DIR)

REQUIRED_DATA = [
    "data/processed/train.parquet",
    "data/processed/test.parquet",
    "data/processed/label_map.json",
]

print("-- Data preflight --")
data_ok = True
for f in REQUIRED_DATA:
    p = Path(f)
    if p.exists():
        print(f"  OK  {f}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  MISSING  {f}")
        data_ok = False

if not data_ok:
    print("WARNING: data files missing. Check Cell 6 output above.")
    print("Cannot train without data -- stopping here.")
    raise FileNotFoundError("Missing data files -- see Cell 6 diagnostics above.")

MANIFEST = Path("outputs/icmlde2026/juliet118/run_manifest.json")
print("\n-- Paper-run manifest --")
if not MANIFEST.exists():
    print("  Not yet created -- will be written by runner (Cell 10).")
else:
    m = json.loads(MANIFEST.read_text())
    print(json.dumps(m, indent=2))
print("Cell 7 done.")


In [ ]:
# ── Cell 8 · Phase 2+3 — Smoke test (dry run, no training) ───────────────────
# Plan Phase 2 checkpoint: seed CLI override works end-to-end.
# Plan Phase 3 checkpoint: per-seed output isolation verified.
import subprocess, sys, os

REPO_DIR = "/kaggle/working/sharp-llm"
os.chdir(REPO_DIR)

cmd = [
    sys.executable, "scripts/run_icmlde_reproducibility.py",
    "--config", "config.yaml",
    "--variants", VARIANT,
    "--seeds", "42",
    "--dry-run",
]
print(f"Dry-run: {' '.join(cmd)}\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise RuntimeError("Dry-run failed — check config.yaml paths and dataset symlinks.")
print("\n✓ Phase 2+3 smoke test passed — CLI, config, and path resolution OK")

In [ ]:
# ── Cell 9 · Phase 4 — Status matrix before training ─────────────────────────
# Plan restart rule: use status.json as the source of truth.
import json
from pathlib import Path

STATUS_PATH   = Path("outputs/icmlde2026/juliet118/status.json")
PLAN_VARIANTS = ["codet5-small", "codet5-base", "codebert-base", "graphcodebert-base"]
PLAN_SEEDS    = [42, 43, 44, 45, 46]

if not STATUS_PATH.exists():
    print("ℹ  No status.json yet — this is the first run.")
else:
    data = json.loads(STATUS_PATH.read_text())
    runs = data.get("runs", {})

    print(f"  {'Variant':<22}", end="")
    for s in PLAN_SEEDS:
        print(f"  seed_{s}", end="")
    print(f"  {'Done':>5}")
    print("  " + "-" * 68)

    total_done = 0
    for variant in PLAN_VARIANTS:
        print(f"  {variant:<22}", end="")
        row_done = 0
        for seed in PLAN_SEEDS:
            state = runs.get(f"seed_{seed}:{variant}", {}).get("state") or "—"
            icon  = "✓" if state == "done" else ("✗" if state == "failed" else "·")
            print(f"  {icon:>7}", end="")
            if state == "done": row_done += 1
        print(f"  {row_done}/5")
        total_done += row_done
    print(f"\n  {total_done}/20 total complete")

In [ ]:
# ── Cell 10 · Phase 4 — Run orchestrator (restartable) ───────────────────────
# Targets only VARIANT set in Cell 4. Skips completed runs automatically.
# If Kaggle 9h limit is hit: download outputs → re-upload dataset → push again.
import subprocess, sys, os, time

REPO_DIR = "/kaggle/working/sharp-llm"
os.chdir(REPO_DIR)

cmd = [
    sys.executable, "scripts/run_icmlde_reproducibility.py",
    "--config",    "config.yaml",
    "--variants",  VARIANT,
    "--seeds",     SEEDS,
    "--continue-on-error",
]

print(f"Variant : {VARIANT}  |  Seeds : {SEEDS}")
print(f"Command : {' '.join(cmd)}")
print("=" * 70)
start = time.time()

result = subprocess.run(cmd)

elapsed = (time.time() - start) / 3600
print(f"\n{'='*70}")
print(f"Wall time : {elapsed:.2f} h")
if result.returncode == 0:
    print(f"✓ Orchestrator complete for: {VARIANT}")
else:
    print(f"⚠ Exit code {result.returncode} — check output above.")
    print("  Partial outputs are safe. Push again after fixing — completed pairs will be skipped.")

In [ ]:
# ── Cell 11 · Phase 3 — Artifact completeness check ──────────────────────────
# Plan §Required Per-Seed Completion Artifacts (all 6 must exist per run).
from pathlib import Path

JULIET_ROOT = Path("outputs/icmlde2026/juliet118")
REQUIRED    = [
    "checkpoints/best.pt", "checkpoints/latest.pt",
    "logs/epoch_metrics.json",
    "evaluation/metrics.json",
    "evaluation/classification_report.txt",
    "evaluation/confusion_pairs.csv",
]

seeds = [int(s.strip()) for s in SEEDS.split(",")]
print(f"{'Run key':<38} {'Artifacts':>10}  Status")
print("-" * 62)
complete = 0
for seed in seeds:
    run_dir = JULIET_ROOT / f"seed_{seed}" / VARIANT
    present = sum(1 for a in REQUIRED if (run_dir / a).exists())
    missing = [a for a in REQUIRED if not (run_dir / a).exists()]
    key     = f"seed_{seed}:{VARIANT}"
    if present == 6:
        print(f"  ✓  {key:<36}  6/6   complete")
        complete += 1
    elif present == 0:
        print(f"  ·  {key:<36}  0/6   not started")
    else:
        print(f"  ~  {key:<36}  {present}/6   partial")
        for m in missing: print(f"         missing: {m}")

print(f"\n  {complete}/5 seeds complete for {VARIANT}")

In [ ]:
# ── Cell 12 · Phase 5+6+7 — Aggregate + significance + summary ───────────────
# Blocked until all 20 runs complete (plan Phase 6 restart rule).
# Run this cell only in the final session after all 4 variants are done.
import json, subprocess, sys
from pathlib import Path

STATUS_PATH = Path("outputs/icmlde2026/juliet118/status.json")

if not STATUS_PATH.exists():
    print("ℹ  No status.json — run Cell 10 first.")
else:
    data   = json.loads(STATUS_PATH.read_text())
    runs   = data.get("runs", {})
    done   = [k for k, r in runs.items() if r.get("state") == "done"]
    print(f"Completion: {len(done)}/20")

    if len(done) < 20:
        missing = [k for k, r in runs.items() if r.get("state") != "done"]
        print(f"⏳ {len(missing)} run(s) remaining — aggregation blocked until all 20 complete.")
        for k in missing: print(f"   · {k}")
    else:
        print("✓ All 20 complete — running aggregation …\n")
        ret = subprocess.run([
            sys.executable, "scripts/aggregate_icmlde_results.py",
            "--config", "config.yaml",
        ])
        if ret.returncode == 0:
            import pandas as pd
            csv = Path("outputs/icmlde2026/juliet118/summary/aggregate_metrics.csv")
            if csv.exists():
                print("\n── Aggregate metrics (mean ± std, 5 seeds) ──────────────")
                print(pd.read_csv(csv).to_string(index=False))
        else:
            print("⚠ Aggregation failed — check seed outputs for missing/invalid files.")

In [ ]:
# ── Cell 13 · Download instructions (run after Cell 10 completes) ─────────────
# Plan §Failure Recovery: outputs must be persisted before session ends.
print(f"""
Session complete for variant: {VARIANT}
Run these commands on your LOCAL machine:
═══════════════════════════════════════════════════════════════════════

# 1. Download this session's outputs:
kaggle kernels output msbasanth/sharp-llm-icmlde-five-seed-runner -p outputs/icmlde2026/

# 2. Re-upload outputs as a new dataset version (enables resume for next session):
kaggle datasets version -p outputs/icmlde2026/ -m "Session complete: {VARIANT}"

# 3. For next session — edit VARIANT in Cell 4, then push:
kaggle kernels push -p colab/

# 4. Poll until done:
kaggle kernels status msbasanth/sharp-llm-icmlde-five-seed-runner

# After all 4 variants are done, run aggregation locally:
python scripts/aggregate_icmlde_results.py --config config.yaml
""")